<a href="https://colab.research.google.com/github/MMM1706/HolsDerGeier_Bot/blob/main/gpu_powered_ctf_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Colab GPU powered cracking workstation

What you can do here:
- You can select which tools you want to be installed (John takes a while)
- You can select which environment you want to be created (connect remotely via SSH or use code snippets right in your browser)
- You can create persistent drive right in your Google Drive

## How to run

Check preferable settings, fill in necessary tokens, connect to GPU Runtime and select Run All (Ctrl+F9)

## Prerequisites
- Ngrok account and auth token (if you want to connect via SSH)
- Working google account (if you want to have persistent data)

# Prepare your environment

In [1]:
#@title Auth tokens

NGROK_AUTH_TOKEN = '3GCA3hVOTJrbO4VC3dXfMW5GC0F_53r5HpGse63VP5oE3DbgZ' #@param {type:"string"}

In [2]:
#@title Tools to install

# Select tools to install
install_hashcat = False #@param {type:"boolean"}
install_john = True #@param {type:"boolean"}
install_hydra = True #@param {type:"boolean"}
collect_wordlists = True #@param {type:"boolean"}


In [3]:
#@title Environment config

# Select environment to set up
environment = "remote_ssh" #@param ["remote_ssh", "in_colab"]
persistent_drive = True #@param {type:"boolean"}

In [4]:
#@title Global variables

# Some global variables
REMOTE_USER_NAME = 'mmm' #@param {type:"string"}
MOUNT_PATH = '/content/drive' #@param {type:"string"}
PERSISTENT_PATH = '{}/My Drive/gpu_powered_storage'.format(MOUNT_PATH)
if persistent_drive:
  STORAGE_PATH = PERSISTENT_PATH
else:
  STORAGE_PATH = '/content/gpu_powered'
HASHCAT_HASH_PATH = "{}/hashcat".format(STORAGE_PATH)
JOHN_PATH = "{}/john".format(STORAGE_PATH)
HYDRA_PATH = "{}/hydra".format(STORAGE_PATH)

In [5]:
#@title Google Drive authorization
#@markdown Simply run this cell and authorize notebook
from google.colab import drive
if persistent_drive:
  drive.mount(MOUNT_PATH)
else:
  pass


Mounted at /content/drive


In [6]:
#@title Create storage
from os import makedirs, path

def create_dir(filepath):
  if not path.exists(filepath):
    makedirs(filepath)
    if path.exists(filepath):
      print("Successfully created {}".format(filepath))
    else:
      print("Cannot create {}".format(filepath))
  else:
    print("Already exists {} - skipping".format(filepath))

create_dir(STORAGE_PATH)

!rm -f /persistent && ln -sf '$STORAGE_PATH' /persistent

Already exists /content/drive/My Drive/gpu_powered_storage - skipping


# Setup your environment with previously selected tools

## Install tools and wordlists

In [7]:
#@title Hashcat installation
#@markdown This script installs hashcat and a few rules for it
out = !hashcat --version
!ln -s $HASHCAT_HASH_PATH /root/.hashcat 2>/dev/null || echo "Already linked"
if not install_hashcat:
  print("Skipped")
elif 'command not found' in ''.join(out) and install_hashcat:
  print("Installing hashcat")
  create_dir(HASHCAT_HASH_PATH)
  !apt -y -qq install cmake build-essential &>/dev/null
  !apt -y -qq install checkinstall git &>/dev/null
  !git clone --quiet https://github.com/hashcat/hashcat.git /tmp/hashcat 2>/dev/null || echo "Already cloned"
  !cd /tmp/hashcat && \
    git submodule update --quiet --init && \
    make -s && make -s install
  print("Hashcat installed")
  !cp -r /tmp/hashcat/rules $HASHCAT_HASH_PATH/rules
  !cp -r /tmp/hashcat/tools $HASHCAT_HASH_PATH/tools
  !git clone --quiet https://github.com/rarecoil/pantagrule.git && \
  cd pantagrule/rules/original && \
  ls && *.gz && mv ./* $HASHCAT_HASH_PATH/rules/ && \
  cd pantagrule/rules/royce && \
  ls && *.gz && mv ./* $HASHCAT_HASH_PATH/rules/ && \
  rm -rf pantagrule
  print(f"Rules and tools available at: {HASHCAT_HASH_PATH}")
else:
  print("Hashcat installed")

out = !hashcat --version
if 'command not found' in ''.join(out) and install_hashcat:
  print("Failed to install hashcat")
else:
  print("Installed hashcat version {}".format(''.join(out)))

Already linked
Skipped
Installed hashcat version /bin/bash: line 1: hashcat: command not found


In [8]:
#@title JohnTheRipper installation
out = !john --version
if not install_john:
  print("Skipped")
elif 'command not found' in ''.join(out) and install_john:
  print("Installing John")
  # !apt-get -y -qq install john &>/dev/null
  # From source - errors
  create_dir(JOHN_PATH)
  src = JOHN_PATH + '/' + 'src'
  create_dir(src)
  !apt-get -y -qq install cmake build-essential libssl-dev zlib1g-dev
  !apt-get -y -qq install yasm pkg-config libgmp-dev libpcap-dev libbz2-dev
  !apt-get -y -qq install nvidia-opencl-dev
  !cd '$src' && \
     git clone --quiet https://github.com/openwall/john -b bleeding-jumbo john 2>/dev/null || echo "Already cloned"
  !cd '$src/john/src' && \
     /bin/bash configure && make -s clean && make -sj4
  !ln -s $src/john/run /usr/share/john
  !ln -s /usr/bin/python3 /bin/python
else:
  print("John installed")

Installing John
Already exists /content/drive/My Drive/gpu_powered_storage/john - skipping
Already exists /content/drive/My Drive/gpu_powered_storage/john/src - skipping
(Reading database ... 122403 files and directories currently installed.)
Removing r-base-dev (4.6.0-4.2204.0) ...
dpkg: pkgconf: dependency problems, but removing anyway as you requested:
 libsndfile1-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libmkl-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libglib2.0-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libfontconfig-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.

Removing pkgconf (1.8

In [9]:
#@title Hydra installation
out = !hydra 1>/dev/null
if not install_hydra:
  print("Skipped")
elif 'command not found' in ''.join(out) and install_hydra:
  print("Installing Hydra")
  create_dir(HYDRA_PATH)
  !apt-get -y -qq install cmake build-essential &>/dev/null
  !apt-get -y -qq install checkinstall git &>/dev/null
  !cd '$HYDRA_PATH' && \
    git clone --quiet https://github.com/vanhauser-thc/thc-hydra.git 2>/dev/null || echo "Already cloned"
  !cd '$HYDRA_PATH/thc-hydra' && \
    /bin/bash configure && make -s && make -s install
else:
  print("Hydra installed")

Installing Hydra
Already exists /content/drive/My Drive/gpu_powered_storage/hydra - skipping
/bin/bash: line 1: cd: $HYDRA_PATH: No such file or directory
Already cloned
/bin/bash: line 1: cd: $HYDRA_PATH/thc-hydra: No such file or directory


In [10]:
#@title Collect most popular wordlists
if collect_wordlists:
  print("Collecting wordlists")
  w_path = '/usr/share/wordlists'
  !mkdir -p $w_path
  # Rockyou
  !ls $w_path/rockyou.txt &>/dev/null || wget -q -P $w_path https://github.com/brannondorsey/naive-hashcat/releases/download/data/rockyou.txt

  print(f"Wordlists collected to {w_path}")

Wordlists collected to /usr/share/wordlists


# Configure connections

This part below is created with a little help from https://github.com/semihucann/hash_cracking_with_gpu

In [11]:
#@title Auto colab reconnect

#@markdown Run this to enable colab env autoreconnect

from IPython.display import display, HTML
js = ("""\
<script>
  function ConnectButton(){
    try {
      document.querySelector('#top-toolbar > colab-connect-button')
      .shadowRoot.querySelector('#connect').click();
      console.log("Connect pushed");
    } catch (error) {
      console.log("Already connected - skipping reconnect")
    }
  }
  setInterval(ConnectButton,30000);
</script>
""")
display(HTML(js))

In [12]:
#@title Ngrok installation

if environment == 'remote_ssh':
  # Install Ngrok
  ngrok_path = '/opt/ngrok'
  !mkdir -p $ngrok_path && wget -q -P $ngrok_path https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
  !ls $ngrok_path/ngrok &>/dev/null || ( cd $ngrok_path && unzip -qq -n $ngrok_path/ngrok-stable-linux-amd64.zip && rm -rf $ngrok_path/ngrok-stable-linux-amd64.zip )
  # Connect to ngrok
  !$ngrok_path/ngrok authtoken $NGROK_AUTH_TOKEN

Authtoken saved to configuration file: /root/.ngrok2/ngrok.yml


In [13]:
#@title SSH setup
if environment == 'remote_ssh':

  import random, string, json, requests
  new_user_password = ''.join(random.choice(string.ascii_letters + string.digits) for i in range(15))

  # SSH
  !apt-get -y -qq install -o=Dpkg::Use-Pty=0 openssh-server pwgen &>/dev/null
  !id -u $REMOTE_USER_NAME &>/dev/null || ( mkdir -p '/home/$REMOTE_USER_NAME' && \
    useradd -d '/home/$REMOTE_USER_NAME' -s /bin/bash $REMOTE_USER_NAME && \
    usermod -aG sudo $REMOTE_USER_NAME )
  !echo $REMOTE_USER_NAME:$new_user_password | chpasswd

  !mkdir -p /var/run/sshd
  !echo "PermitRootLogin yes" >> /etc/ssh/sshd_config
  !echo "PasswordAuthentication yes" >> /etc/ssh/sshd_config
  !echo "LD_LIBRARY_PATH=/usr/lib64-nvidia" >> /root/.bashrc
  !echo "export LD_LIBRARY_PATH" >> /root/.bashrc

  #Run sshd
  get_ipython().system_raw('/usr/sbin/sshd -D &')
else:
  print("This machine is fully set up - use it wisely")

In [ ]:
 #@title Run environment

if environment == 'remote_ssh':
  import time, requests

  # Start ngrok for ssh
  tmp = !pgrep ngrok
  if tmp:
    !kill -9 {' '.join(tmp)}

  get_ipython().system_raw(f'{ngrok_path}/ngrok tcp 22 &')

  print("Warte auf Ngrok-Tunnel...")
  address = ""
  for i in range(10):
    time.sleep(3)
    try:
      r = requests.get('http://localhost:4040/api/tunnels')
      if r.ok:
        tunnels = r.json().get('tunnels', [])
        if tunnels:
          address = tunnels[0]['public_url'].replace('tcp://', '')
          break
    except:
      continue

  if address:
    ip, port = address.split(':')
    print(f"\nVerbindung erfolgreich hergestellt!")
    print(f"Nutzen Sie diesen Befehl im Terminal:")
    print(f"\033[1;32mssh {REMOTE_USER_NAME}@{ip} -p {port}\033[0m")
    print(f"Passwort: {new_user_password}")
  else:
    print("Fehler: Der Tunnel konnte nicht gestartet werden. Prüfen Sie Ihren NGROK_AUTH_TOKEN.")
else:
  print("Machine is ready (In-Colab mode).")

Warte auf Ngrok-Tunnel...


# Run what you want!

In [15]:
# !hashcat -m 0 --benchmark
# !/usr/share/john/john
# !hydra --version

# Remove connections

In [16]:
#@markdown If you want to clean connections, check field below and run this cell (remember to uncheck this after run)
remove_all = False #@param {type:"boolean"}

if remove_all:
  tmp = !pgrep ngrok
  id = ''.join(tmp)
  !kill -9 $id &>/dev/null